# Setup

In [2]:
from dotenv import load_dotenv

# Loads OPENAI_API_KEY from the existing Agent_Only .env (cross-repo,
# not committed in this project). Alternatively export OPENAI_API_KEY
# yourself or create a local .env from .env.example.
load_dotenv("/home/ubuntu/information-extraction/Agent_Only/.env")

True

In [3]:
# Run once if the OpenAI SDK is not installed in this environment.
# %pip install openai

import json
import os
from getpass import getpass
from textwrap import dedent

try:
    from openai import OpenAI
except ImportError as exc:
    raise ImportError("Install the OpenAI SDK first by running: %pip install openai") from exc

In [4]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: " )

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
MODEL

'gpt-4.1-mini'

# Prompt

In [ ]:
GLINER2_GUIDELINE_PROMPT = dedent("""
You are a GLiNER2 label schema designer for clinical information extraction.

GLiNER2 (https://github.com/fastino-ai/GLiNER2) is a single-pass zero-shot model that
performs four extraction tasks from one schema: entity extraction, structured (JSON)
extraction, text classification, and relation extraction. Your job is to read a
free-text description of what a user wants extracted from clinical notes, decide --
for every requested piece of information -- which of these four task types it
belongs to, and emit a schema as JSON.

Do not provide medical advice or diagnose the patient. Only design extraction labels.

## The four task types, and how to choose between them

1. entities -- a standalone span of text that can be pulled out on its own,
   independent of any other attribute (e.g. a drug name, a symptom, a test name,
   a body site). Use this whenever the requested item is not tightly coupled to
   other attributes describing the same instance, and whenever it needs to
   participate in a relation (relations only connect entities -- see below).

2. structures -- a named, multi-field record for when two or more requested
   attributes describe the SAME instance/occurrence and should stay grouped
   together (e.g. a medication mention's dosage + frequency + route all describe
   one medication event; a biomarker's value + unit describe one biomarker
   result). GLiNER2 automatically extracts every instance of a structure found in
   the text, so prefer a structure whenever the request implies "for each X, also
   capture its Y and Z". A field's value can be constrained to a fixed vocabulary
   with `choices` (e.g. a severity field with choices mild/moderate/severe).

3. classifications -- a closed set of labels assigned to the whole document (or a
   whole note), not to one particular instance (e.g. document_type, overall
   urgency, overall sentiment). If a closed set of choices instead describes ONE
   specific instance found in the text (e.g. the stage of one particular cancer
   mention, the severity of one particular symptom), model it as a `choices`
   field inside a structure instead -- do not use classifications for per-instance
   attributes.

4. relations -- a named, directional relationship between two entities already
   defined in this same schema's `entities` list (e.g. "MEDICATION treats
   PROBLEM", "TEST diagnoses PROBLEM"). Relations only ever connect entity spans,
   never structure names or classification labels -- if the request implies a
   relationship involving a structured/grouped concept, also add a plain entity
   for that concept so it can be the head or tail of the relation. Phrase each
   relation as "<SourceLabel> <verb phrase> <TargetLabel>", referencing labels you
   defined in `entities`.

## Attribute contract

Every entity, and every structure field, may carry:
- `dtype`: "str" for a single best span (use when only one value is expected per
  instance, e.g. a name), or "list" for zero or more spans (default; use when
  if we have provided a choice list in structure fields).
- `description`: a short natural-language clarification of what counts as a match
  -- always include one, since it materially improves extraction accuracy.

Structure fields (not entities) may additionally carry:
- `choices` (optional): a fixed list of allowed values, when the value should be
  normalized to a closed vocabulary (e.g. `["mild", "moderate", "severe"]`). Omit
  this key entirely when the value is free text. Entities never carry `choices`.

Classification tasks carry `labels` (the fixed label set) instead of `choices`,
and may set `"multi_label": true` when more than one label can apply at once
(omit for single-label, which is the default).

Do not include threshold/confidence values anywhere in the output -- thresholds
are configured separately downstream, per task type.

## Output contract

Return a single valid JSON object only -- no markdown fences, no prose before or
after -- with exactly these four keys:

{
  "entities": [
    {"label": "<NAME>", "dtype": "str", "description": "<...>"}
  ],
  "structures": [
    {
      "name": "<structure_name>",
      "fields": [
        {"name": "<field_name>", "dtype": "str", "description": "<...>"},
        {"name": "<field_name>", "dtype": "str"|"list", "description": "<...>", "choices": ["...", ...]}               
      ]
    }
  ],
  "classifications": [
    {"task": "<task_name>", "labels": ["...", ...], "multi_label": true|false}
  ],
  "relations": [
    "<SourceLabel> <verb phrase> <TargetLabel>"
  ]
}

Omit the `choices`/`multi_label` keys on any item that doesn't need them -- don't
emit them as null or empty. Any of the four top-level arrays may be empty ([]) if
the user's request doesn't call for that task type, but all four keys must always
be present.

## Worked example

User extraction request:
"Pull out the patient's allergies and the drugs prescribed for them, including
dose and how often they're taken. Flag whether the note is a discharge summary,
progress note, or radiology report."

Expected schema:
{
  "entities": [
    {"label": "ALLERGY", "dtype": "str", "description": "A substance or drug the patient is allergic to"},
    {"label": "MEDICATION", "dtype": "str", "description": "Name of a prescribed drug"}
  ],
  "structures": [
    {
      "name": "medication_item",
      "fields": [
        {"name": "dosage", "dtype": "str", "description": "Dose amount and unit, e.g. 500mg"},
        {"name": "frequency", "dtype": "str", "description": "How often the drug is taken"}
      ]
    }
  ],
  "classifications": [
    {"task": "document_type", "labels": ["Discharge Summary", "Progress Note", "Radiology Report"]}
  ],
  "relations": [
    "MEDICATION prescribed_for ALLERGY"
  ]
}

Note how "dose and how often" stayed grouped into one `medication_item` structure
(they describe the same prescription instance) rather than becoming separate
entities; "discharge summary / progress note / radiology report" became a
document-level classification, not a per-instance choices field; and MEDICATION
was kept as its own entity (in addition to the medication_item structure) purely
so it could be the source of the `prescribed_for` relation.
""").strip()

print(GLINER2_GUIDELINE_PROMPT[:1200] + "\n...")

You are designing a schema for GLiNER2.

Your job is to convert a user's extraction request plus the input clinical note into a compact schema that can be fed into GLiNER2 for entity extraction, structured extraction, classification, and relation extraction.

Important assumptions about GLiNER2:
- GLiNER2 is schema-driven. Entity labels, descriptions, structures, classifications, and relations are part of the task definition.
- It works best when labels are semantically clear and descriptions are concise.
- It is better to define what a label means than to write long extraction instructions.
- Nearby labels that overlap semantically can confuse extraction, so descriptions should help separate them.
- Structures should group fields that belong to the same local record or mention.
- Very wide structures should be avoided unless the fields clearly belong together.
- Relations should only be included when they link meaningful standalone entities.
- Keep the schema grounded in the user's re

# AI Prediction

In [12]:
user_request = (
        "Extract all 'diagnoses', 'symptoms', 'cancer mentions',  'body site', 'biomarkers', 'treatments', 'medications', 'tests', entities."
        "'cancer stage' should be stage I, stage II, stage III, stage IV, or unknown. "
        "also extract structure of  biomarker name, biomarker results, and related date with biomarker. "
        "extract relation between Medication and Adverse Event information. "
    )

print(user_request)

Extract all 'diagnoses', 'symptoms', 'cancer mentions',  'body site', 'biomarkers', 'treatments', 'medications', 'tests', entities.'cancer stage' should be stage I, stage II, stage III, stage IV, or unknown. also extract structure of  biomarker name, biomarker results, and related date with biomarker. extract relation between Medication and Adverse Event information. 


In [13]:
merged_prompt = dedent(f"""
{GLINER2_GUIDELINE_PROMPT}

USER EXTRACTION REQUEST:
{user_request}

Create the best GLiNER2 schema for this request. Return a single valid JSON object
only, with exactly the keys entities, structures, classifications, relations as
defined above.
""").strip()

In [14]:
response = client.responses.create(
    model=MODEL,
    instructions=(
        "You generate precise GLiNER2 schemas for medical information extraction. "
        "Return a single valid JSON object only, with exactly the keys entities, "
        "structures, classifications, relations. Do not include markdown fences or "
        "explanatory text."
    ),
    input=merged_prompt,
    temperature=0.1,
)

api_response_text = response.output_text
print(api_response_text)

{
  "entities": [
    {
      "label": "DIAGNOSIS",
      "dtype": "str",
      "description": "Medical condition diagnosed in the patient"
    },
    {
      "label": "SYMPTOM",
      "dtype": "str",
      "description": "Patient-reported or observed symptom"
    },
    {
      "label": "CANCER_MENTION",
      "dtype": "str",
      "description": "Mention of cancer type or cancer-related diagnosis"
    },
    {
      "label": "BODY_SITE",
      "dtype": "str",
      "description": "Anatomical location or body part"
    },
    {
      "label": "TREATMENT",
      "dtype": "str",
      "description": "Therapeutic intervention or procedure"
    },
    {
      "label": "MEDICATION",
      "dtype": "str",
      "description": "Drug or pharmaceutical agent"
    },
    {
      "label": "TEST",
      "dtype": "str",
      "description": "Diagnostic or laboratory test"
    },
    {
      "label": "ADVERSE_EVENT",
      "dtype": "str",
      "description": "Negative or unintended effect related 

In [9]:
response = client.responses.create(
    model=MODEL,
    instructions=(
        "You generate precise GLiNER2 schemas for medical information extraction. "
        "Return a single valid JSON object only, with exactly the keys entities, "
        "structures, classifications, relations. Do not include markdown fences or "
        "explanatory text."
    ),
    input=merged_prompt,
    temperature=0.1,
)

api_response_text = response.output_text
print(api_response_text)

{
  "entities": [
    {
      "label": "DIAGNOSIS",
      "dtype": "str",
      "description": "A medical diagnosis or condition mentioned in the text"
    },
    {
      "label": "SYMPTOM",
      "dtype": "str",
      "description": "A symptom or sign reported by or observed in the patient"
    },
    {
      "label": "CANCER_MENTION",
      "dtype": "str",
      "description": "A mention of cancer or malignancy in the text"
    },
    {
      "label": "BODY_SITE",
      "dtype": "str",
      "description": "An anatomical location or body site referenced in the clinical note"
    },
    {
      "label": "BIOMARKER",
      "dtype": "str",
      "description": "Name of a biomarker or lab test marker"
    },
    {
      "label": "TREATMENT",
      "dtype": "str",
      "description": "A treatment or therapeutic intervention mentioned"
    },
    {
      "label": "MEDICATION",
      "dtype": "str",
      "description": "Name of a medication or drug mentioned"
    },
    {
      "label": "

# Parse

In [9]:
import json
try:
    parsed_response = json.loads(api_response_text)
    print(json.dumps(parsed_response, indent=2))
except json.JSONDecodeError:
    print("The API response was not valid JSON. Raw response:")
    print(api_response_text)

{
  "entities": [
    {
      "label": "DIAGNOSIS",
      "dtype": "str",
      "description": "Medical condition diagnosis"
    },
    {
      "label": "SYMPTOM",
      "dtype": "str",
      "description": "Reported symptom or sign"
    },
    {
      "label": "CANCER_MENTION",
      "dtype": "str",
      "description": "Mention of cancer"
    },
    {
      "label": "BODY_SITE",
      "dtype": "str",
      "description": "Anatomical location"
    },
    {
      "label": "TREATMENT",
      "dtype": "str",
      "description": "Therapeutic intervention"
    },
    {
      "label": "MEDICATION",
      "dtype": "str",
      "description": "Drug or medication name"
    },
    {
      "label": "TEST",
      "dtype": "str",
      "description": "Diagnostic or lab test"
    },
    {
      "label": "ADVERSE_EVENT",
      "dtype": "str",
      "description": "Negative reaction or side effect"
    }
  ],
  "structures": [
    {
      "name": "biomarker",
      "fields": [
        {
          "n

In [10]:
eval(api_response_text)

{'entities': [{'label': 'DIAGNOSIS',
   'dtype': 'str',
   'description': 'Medical condition diagnosis'},
  {'label': 'SYMPTOM',
   'dtype': 'str',
   'description': 'Reported symptom or sign'},
  {'label': 'CANCER_MENTION',
   'dtype': 'str',
   'description': 'Mention of cancer'},
  {'label': 'BODY_SITE', 'dtype': 'str', 'description': 'Anatomical location'},
  {'label': 'TREATMENT',
   'dtype': 'str',
   'description': 'Therapeutic intervention'},
  {'label': 'MEDICATION',
   'dtype': 'str',
   'description': 'Drug or medication name'},
  {'label': 'TEST', 'dtype': 'str', 'description': 'Diagnostic or lab test'},
  {'label': 'ADVERSE_EVENT',
   'dtype': 'str',
   'description': 'Negative reaction or side effect'}],
 'structures': [{'name': 'biomarker',
   'fields': [{'name': 'name',
     'dtype': 'str',
     'description': 'Biomarker name'},
    {'name': 'result', 'dtype': 'str', 'description': 'Biomarker test result'},
    {'name': 'date',
     'dtype': 'str',
     'description': '

In [ ]:
# --- Control test 1: schema contract ---
# parsed_response must match the exact shape build_multitask_pipeline.ipynb's
# `config` dict uses (see that notebook's "## 1. Config" cell).

EXPECTED_KEYS = {"entities", "structures", "classifications", "relations"}

assert isinstance(parsed_response, dict), f"expected a dict, got {type(parsed_response)}"
assert EXPECTED_KEYS.issubset(parsed_response.keys()), (
    f"missing keys: {EXPECTED_KEYS - parsed_response.keys()}"
)

entity_labels = set()
for e in parsed_response["entities"]:
    assert isinstance(e, dict) and e.get("label"), f"entity missing label: {e}"
    assert e.get("dtype", "list") in ("str", "list"), f"entity {e['label']} has invalid dtype: {e.get('dtype')}"
    assert "choices" not in e, f"entity {e['label']} must not use choices (GLiNER2 entities don't support it): {e}"
    entity_labels.add(e["label"])

for s in parsed_response["structures"]:
    assert s.get("name"), f"structure missing name: {s}"
    fields = s.get("fields")
    assert isinstance(fields, list) and fields, f"structure {s.get('name')} needs at least one field: {s}"
    for f in fields:
        assert f.get("name"), f"field missing name in structure {s['name']}: {f}"
        if "choices" in f:
            assert isinstance(f["choices"], list) and all(isinstance(c, str) for c in f["choices"]), (
                f"field {f['name']} choices must be list[str]: {f['choices']}"
            )
        else:
            assert f.get("dtype", "list") in ("str", "list"), f"field {f['name']} has invalid dtype: {f.get('dtype')}"

for c in parsed_response["classifications"]:
    assert c.get("task"), f"classification missing task: {c}"
    assert isinstance(c.get("labels"), list) and c["labels"], f"classification {c.get('task')} needs labels: {c}"
    if "multi_label" in c:
        assert isinstance(c["multi_label"], bool), f"classification {c['task']} multi_label must be bool"

for r in parsed_response["relations"]:
    assert isinstance(r, str) and r.strip(), f"relation must be a non-empty string: {r!r}"
    assert any(label in r.split() for label in entity_labels), (
        f"relation {r!r} doesn't reference any defined entity label {entity_labels}"
    )

print(f"[PASS] schema contract: {len(parsed_response['entities'])} entities, "
      f"{len(parsed_response['structures'])} structures, "
      f"{len(parsed_response['classifications'])} classifications, "
      f"{len(parsed_response['relations'])} relations.")

[PASS] schema contract: 9 entities, 2 structures, 0 classifications, 1 relations.


In [22]:
parsed_response

{'entities': [{'label': 'DIAGNOSIS',
   'dtype': 'str',
   'description': 'A medical diagnosis or condition mentioned in the text'},
  {'label': 'SYMPTOM',
   'dtype': 'str',
   'description': 'A symptom or sign reported by or observed in the patient'},
  {'label': 'CANCER_MENTION',
   'dtype': 'str',
   'description': 'A mention of cancer or malignancy in the text'},
  {'label': 'BODY_SITE',
   'dtype': 'str',
   'description': 'An anatomical location or body site referenced in the clinical note'},
  {'label': 'BIOMARKER',
   'dtype': 'str',
   'description': 'Name of a biomarker or lab test marker'},
  {'label': 'TREATMENT',
   'dtype': 'str',
   'description': 'A treatment or therapeutic intervention mentioned'},
  {'label': 'MEDICATION',
   'dtype': 'str',
   'description': 'Name of a medication or drug mentioned'},
  {'label': 'TEST',
   'dtype': 'str',
   'description': 'A diagnostic or laboratory test mentioned'},
  {'label': 'ADVERSE_EVENT',
   'dtype': 'str',
   'description':